# 🏥 의료 데이터 분석
## 01. 데이터 개요 (Data Overview)

---

### 📋 프로젝트 소개
본 프로젝트는 GitHub의 `Basic_Health_Care` 오픈소스 의료 기록 데이터를 활용하여 환자의 입원 패턴과 진단 경향성을 분석합니다.

* **데이터 출처:** [csbond007 GitHub - Basic Health Care Repository](https://github.com/csbond007/Basic_Health_Care)
* **데이터 성격:** - 오픈소스 시뮬레이터인 **Synthea**를 통해 생성된 가상 의료 데이터셋입니다.
    - 실제 임상 가이드라인을 준수하여 생성되었으므로, 환자의 진료 경로(Clinical Pathway)와 검사 결과 간의 상관관계가 논리적으로 구성되어 있습니다.
- **분석 적합성:**
    - 단순한 수치를 넘어 이상치 발생 빈도, 검사 주기 등 **복합적인 피처 엔지니어링** 연습에 매우 적합한 구조를 가지고 있습니다.

**데이터셋 구성:**
- `PatientCorePopulatedTable`: 환자 인구통계 정보
- `AdmissionsCorePopulatedTable`: 입원 기록
- `AdmissionsDiagnosesCorePopulatedTable`: 진단 정보
- `LabsCorePopulatedTable`: 검사 결과

---
## 1. 환경 설정

In [1]:
# 필수 라이브러리
import os
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
import warnings

warnings.filterwarnings('ignore')
plt.style.use('default')
plt.rcParams['figure.figsize'] = (12, 6)

# 한글 폰트 설정 (Windows)
plt.rcParams['font.family'] = 'Malgun Gothic'
plt.rcParams['axes.unicode_minus'] = False

# 출력 옵션
pd.set_option('display.max_columns', None)
pd.set_option('display.max_rows', 100)

---
## 2. 데이터 로드

GitHub 원본 데이터를 로컬로 다운로드하여 사용합니다.

In [4]:
import requests

# 데이터 URL
URLS = {
    "patients": "https://raw.githubusercontent.com/tashydean/Basic_Health_Care/refs/heads/master/data/PatientCorePopulatedTable.txt",
    "admissions": "https://raw.githubusercontent.com/tashydean/Basic_Health_Care/refs/heads/master/data/AdmissionsCorePopulatedTable.txt",
    "diagnoses": "https://raw.githubusercontent.com/tashydean/Basic_Health_Care/refs/heads/master/data/AdmissionsDiagnosesCorePopulatedTable.txt",
    "labs": "https://raw.githubusercontent.com/tashydean/Basic_Health_Care/refs/heads/master/data/LabsCorePopulatedTable.txt"
}

# 데이터 다운로드 함수
def load_data(url, columns):
    """GitHub에서 TSV 데이터 로드"""
    response = requests.get(url)
    from io import StringIO
    df = pd.read_csv(StringIO(response.text), sep='\t', skiprows=1, header=None, names=columns)
    return df

# 각 데이터셋 로드
print("📥 데이터 로딩 중...")

df_patients = load_data(
    URLS['patients'],
    ['PatientID', 'PatientGender', 'PatientDateOfBirth', 'PatientRace', 
     'PatientMaritalStatus', 'PatientLanguage', 'PatientPopulationPercentageBelowPoverty']
)

df_admissions = load_data(
    URLS['admissions'],
    ['PatientID', 'AdmissionID', 'AdmissionStartDate', 'AdmissionEndDate']
)

df_diagnoses = load_data(
    URLS['diagnoses'],
    ['PatientID', 'AdmissionID', 'PrimaryDiagnosisCode', 'PrimaryDiagnosisDescription']
)

df_labs = load_data(
    URLS['labs'],
    ['PatientID', 'AdmissionID', 'LabName', 'LabValue', 'LabUnits', 'LabDateTime']
)

print("✅ 데이터 로드 완료!")

📥 데이터 로딩 중...
✅ 데이터 로드 완료!


---
## 3. 데이터 기본 정보

각 데이터셋의 크기와 구조를 확인합니다.

In [5]:
datasets = {
    'Patients (환자 정보)': df_patients,
    'Admissions (입원 기록)': df_admissions,
    'Diagnoses (진단 정보)': df_diagnoses,
    'Labs (검사 결과)': df_labs
}

print("="*80)
print("📊 데이터셋 크기")
print("="*80)
for name, df in datasets.items():
    print(f"{name:30s} | Rows: {df.shape[0]:>6,} | Columns: {df.shape[1]:>2}")
print("="*80)

📊 데이터셋 크기
Patients (환자 정보)               | Rows:    100 | Columns:  7
Admissions (입원 기록)             | Rows:    372 | Columns:  4
Diagnoses (진단 정보)              | Rows:    372 | Columns:  4
Labs (검사 결과)                   | Rows: 111,483 | Columns:  6


### 3.1 Patients (환자 인구통계)

In [6]:
print("\n👤 환자 데이터 (Patient Demographics)\n")
print(df_patients.info())
print("\n▶ Sample Data:")
df_patients.head()


👤 환자 데이터 (Patient Demographics)

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 100 entries, 0 to 99
Data columns (total 7 columns):
 #   Column                                   Non-Null Count  Dtype  
---  ------                                   --------------  -----  
 0   PatientID                                100 non-null    object 
 1   PatientGender                            100 non-null    object 
 2   PatientDateOfBirth                       100 non-null    object 
 3   PatientRace                              100 non-null    object 
 4   PatientMaritalStatus                     100 non-null    object 
 5   PatientLanguage                          100 non-null    object 
 6   PatientPopulationPercentageBelowPoverty  100 non-null    float64
dtypes: float64(1), object(6)
memory usage: 5.6+ KB
None

▶ Sample Data:


,PatientID,PatientGender,PatientDateOfBirth,PatientRace,PatientMaritalStatus,PatientLanguage,PatientPopulationPercentageBelowPoverty
0,FB2ABB23-C9D0-4D09-8464-49BF0B982F0F,Male,1947-12-28 02:45:40.547,Unknown,Married,Icelandic,18.08
1,64182B95-EB72-4E2B-BE77-8050B71498CE,Male,1952-01-18 19:51:12.917,African American,Separated,English,13.03
2,DB22A4D9-7E4D-485C-916A-9CD1386507FB,Female,1970-07-25 13:04:20.717,Asian,Married,English,6.67
3,6E70D84D-C75F-477C-BC37-9177C3698C66,Male,1979-01-04 05:45:29.580,White,Married,English,16.09
4,C8556CC0-32FC-4CA5-A8CD-9CCF38816167,Female,1921-04-11 11:39:49.197,White,Married,English,18.20


### 3.2 Admissions (입원 기록)

In [7]:
print("\n🏥 입원 데이터 (Admission Records)\n")
print(df_admissions.info())
print("\n▶ Sample Data:")
df_admissions.head()


🏥 입원 데이터 (Admission Records)

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 372 entries, 0 to 371
Data columns (total 4 columns):
 #   Column              Non-Null Count  Dtype 
---  ------              --------------  ----- 
 0   PatientID           372 non-null    object
 1   AdmissionID         372 non-null    int64 
 2   AdmissionStartDate  372 non-null    object
 3   AdmissionEndDate    372 non-null    object
dtypes: int64(1), object(3)
memory usage: 11.8+ KB
None

▶ Sample Data:


,PatientID,AdmissionID,AdmissionStartDate,AdmissionEndDate
0,7A025E77-7832-4F53-B9A7-09A3F98AC17E,7,2011-10-12 14:55:02.027,2011-10-22 01:16:07.557
1,DCE5AEB8-6DB9-4106-8AE4-02CCC5C23741,1,1993-02-11 18:57:04.003,1993-02-24 17:22:29.713
2,DCE5AEB8-6DB9-4106-8AE4-02CCC5C23741,2,2002-11-28 19:06:31.117,2002-12-04 19:14:40.797
3,DCE5AEB8-6DB9-4106-8AE4-02CCC5C23741,3,2011-07-19 18:42:45.287,2011-07-25 04:57:42.053
4,886B5885-1EE2-49F3-98D5-A2F02EB8A9D4,1,1994-12-03 22:20:46.077,1994-12-20 20:24:56.010


### 3.3 Diagnoses (진단 정보)

In [8]:
print("\n💊 진단 데이터 (Diagnosis Information)\n")
print(df_diagnoses.info())
print("\n▶ Sample Data:")
df_diagnoses.head()


💊 진단 데이터 (Diagnosis Information)

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 372 entries, 0 to 371
Data columns (total 4 columns):
 #   Column                       Non-Null Count  Dtype 
---  ------                       --------------  ----- 
 0   PatientID                    372 non-null    object
 1   AdmissionID                  372 non-null    int64 
 2   PrimaryDiagnosisCode         372 non-null    object
 3   PrimaryDiagnosisDescription  372 non-null    object
dtypes: int64(1), object(3)
memory usage: 11.8+ KB
None

▶ Sample Data:


,PatientID,AdmissionID,PrimaryDiagnosisCode,PrimaryDiagnosisDescription
0,80AC01B2-BD55-4BE0-A59A-4024104CF4E9,2,M01.X,Direct infection of joint in infectious and pa...
1,80AC01B2-BD55-4BE0-A59A-4024104CF4E9,3,D65,Disseminated intravascular coagulation [defibr...
2,80AC01B2-BD55-4BE0-A59A-4024104CF4E9,4,C92.1,"Chronic myeloid leukemia, BCR/ABL-positive"
3,80AC01B2-BD55-4BE0-A59A-4024104CF4E9,5,M05.51,Rheumatoid polyneuropathy with rheumatoid arth...
4,6A57AC0C-57F3-4C19-98A1-51135EFBC4FF,1,C91.00,Acute lymphoblastic leukemia not having achiev...


### 3.4 Labs (검사 결과)

In [9]:
print("\n🧪 검사 데이터 (Laboratory Tests)\n")
print(df_labs.info())
print("\n▶ Sample Data:")
df_labs.head()


🧪 검사 데이터 (Laboratory Tests)

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 111483 entries, 0 to 111482
Data columns (total 6 columns):
 #   Column       Non-Null Count   Dtype  
---  ------       --------------   -----  
 0   PatientID    111483 non-null  object 
 1   AdmissionID  111483 non-null  int64  
 2   LabName      111483 non-null  object 
 3   LabValue     111483 non-null  float64
 4   LabUnits     111483 non-null  object 
 5   LabDateTime  111483 non-null  object 
dtypes: float64(1), int64(1), object(4)
memory usage: 5.1+ MB
None

▶ Sample Data:


,PatientID,AdmissionID,LabName,LabValue,LabUnits,LabDateTime
0,1A8791E3-A61C-455A-8DEE-763EB90C9B2C,1,URINALYSIS: RED BLOOD CELLS,1.8,rbc/hpf,1992-07-01 01:36:17.910
1,1A8791E3-A61C-455A-8DEE-763EB90C9B2C,1,METABOLIC: GLUCOSE,103.3,mg/dL,1992-06-30 09:35:52.383
2,1A8791E3-A61C-455A-8DEE-763EB90C9B2C,1,CBC: MCH,35.8,pg,1992-06-30 03:50:11.777
3,1A8791E3-A61C-455A-8DEE-763EB90C9B2C,1,METABOLIC: CALCIUM,8.9,mg/dL,1992-06-30 12:09:46.107
4,1A8791E3-A61C-455A-8DEE-763EB90C9B2C,1,CBC: RED BLOOD CELL COUNT,4.8,m/cumm,1992-07-01 01:31:08.677


---
## 4. 데이터 품질 체크

결측치, 중복 등을 확인합니다.

In [10]:
def data_quality_check(df, name):
    """데이터 품질 체크 함수"""
    print(f"\n{'='*80}")
    print(f"🔍 {name} - 데이터 품질 체크")
    print(f"{'='*80}")
    
    # 결측치
    missing = df.isnull().sum()
    if missing.sum() > 0:
        print("\n⚠️ 결측치:")
        print(missing[missing > 0])
    else:
        print("\n✅ 결측치 없음")
    
    # 중복
    dup_count = df.duplicated().sum()
    print(f"\n🔄 중복 행: {dup_count:,}개")
    
    return missing, dup_count

# 각 데이터셋 체크
for name, df in datasets.items():
    data_quality_check(df, name)


🔍 Patients (환자 정보) - 데이터 품질 체크

✅ 결측치 없음

🔄 중복 행: 0개

🔍 Admissions (입원 기록) - 데이터 품질 체크

✅ 결측치 없음

🔄 중복 행: 0개

🔍 Diagnoses (진단 정보) - 데이터 품질 체크

✅ 결측치 없음

🔄 중복 행: 0개

🔍 Labs (검사 결과) - 데이터 품질 체크

✅ 결측치 없음

🔄 중복 행: 0개


---
## 5. 데이터 요약 통계

주요 숫자형 변수의 기초 통계량을 확인합니다.

In [11]:
# 환자별 입원 횟수
admissions_per_patient = df_admissions.groupby('PatientID').size()

print("📈 환자별 입원 횟수 통계")
print(f"평균: {admissions_per_patient.mean():.2f}회")
print(f"중앙값: {admissions_per_patient.median():.1f}회")
print(f"최대: {admissions_per_patient.max()}회")
print(f"\n총 환자 수: {df_patients.shape[0]:,}명")
print(f"총 입원 기록: {df_admissions.shape[0]:,}건")
print(f"총 진단 기록: {df_diagnoses.shape[0]:,}건")
print(f"총 검사 기록: {df_labs.shape[0]:,}건")

📈 환자별 입원 횟수 통계
평균: 3.72회
중앙값: 3.5회
최대: 7회

총 환자 수: 100명
총 입원 기록: 372건
총 진단 기록: 372건
총 검사 기록: 111,483건


---
## 6. 데이터 간 관계

각 테이블의 연결 관계를 확인합니다.

In [14]:
print("🔗 데이터 관계 확인\n")

# PatientID 일치 여부
patients_in_admissions = df_admissions['PatientID'].isin(df_patients['PatientID']).all()
print(f"✓ 모든 입원 기록의 환자가 Patients 테이블에 존재: {patients_in_admissions}")

🔗 데이터 관계 확인

✓ 모든 입원 기록의 환자가 Patients 테이블에 존재: True


In [15]:
print("🔗 데이터 관계 정밀 검증 (Composite Key 기준)\n")

# 1. 검증용 도우미 함수 (환자ID와 입원ID 조합 생성)
def get_combined_keys(df):
    return df['PatientID'].astype(str) + "_" + df['AdmissionID'].astype(str)

# 각 테이블의 복합 키 생성
keys_admissions = set(get_combined_keys(df_admissions))
keys_diagnoses = set(get_combined_keys(df_diagnoses))
keys_labs = set(get_combined_keys(df_labs))

# 2. 진단(Diagnosis) -> 입원(Admissions) 체크
# 모든 진단 기록이 실제 입원 기록(Admissions)에 기반하는가?
missing_diag = keys_diagnoses - keys_admissions
is_diag_valid = len(missing_diag) == 0

print(f"🔍 [진단 -> 입원] 검증")
print(f"  - 모든 진단의 입원 기록이 Admissions에 존재: {is_diag_valid}")
if not is_diag_valid:
    print(f"  - ⚠️ 누락된 입원 키 샘플 (첫 5개): {list(missing_diag)[:5]}")
    print(f"  - ⚠️ 누락 건수: {len(missing_diag)} 건")

print("-" * 40)

# 3. 검사(Labs) -> 입원(Admissions) 체크
# 모든 검사 기록이 실제 입원 기록(Admissions)에 기반하는가?
missing_labs = keys_labs - keys_admissions
is_labs_valid = len(missing_labs) == 0

print(f"🔍 [검사 -> 입원] 검증")
print(f"  - 모든 검사의 입원 기록이 Admissions에 존재: {is_labs_valid}")
if not is_labs_valid:
    print(f"  - ⚠️ 누락된 입원 키 샘플 (첫 5개): {list(missing_labs)[:5]}")
    print(f"  - ⚠️ 누락 건수: {len(missing_labs)} 건")

🔗 데이터 관계 정밀 검증 (Composite Key 기준)

🔍 [진단 -> 입원] 검증
  - 모든 진단의 입원 기록이 Admissions에 존재: True
----------------------------------------
🔍 [검사 -> 입원] 검증
  - 모든 검사의 입원 기록이 Admissions에 존재: True


---
## 7. 요약

**데이터셋 개요:**
- 총 100명의 환자 데이터
- 372건의 입원 기록
- 111,483건의 검사 결과
- 결측치 없음, 중복 없음
- 데이터 간 참조 무결성 유지됨

**다음 단계:** 02_feature_engineering.ipynb에서 본격적인 전처리와 피처 생성을 진행합니다.